# Food Delivery

Data cleaning and extraction

https://www.kaggle.com/datasets/rajatkumar30/food-delivery-time

In [15]:
import pandas as pd

In [16]:
df = pd.read_csv("deliverytime.csv")

In [17]:
df.head()

,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Type_of_order,Type_of_vehicle,Time_taken(min)
0,4607,INDORES13DEL02,37,4.9,22.745049,75.892471,22.765049,75.912471,Snack,motorcycle,24
1,B379,BANGRES18DEL02,34,4.5,12.913041,77.683237,13.043041,77.813237,Snack,scooter,33
2,5D6D,BANGRES19DEL01,23,4.4,12.914264,77.678400,12.924264,77.688400,Drinks,motorcycle,26
3,7A6A,COIMBRES13DEL02,38,4.7,11.003669,76.976494,11.053669,77.026494,Buffet,motorcycle,21
4,70A2,CHENRES12DEL01,32,4.6,12.972793,80.249982,13.012793,80.289982,Snack,scooter,30


## Treat Latitude and Longitude

In [18]:
import folium
from folium.plugins import FastMarkerCluster

m = folium.Map(location=[20, 0], zoom_start=2)

restaurant_points = df[
    ["Restaurant_latitude", "Restaurant_longitude"]
].dropna().values.tolist()

delivery_points = df[
    ["Delivery_location_latitude", "Delivery_location_longitude"]
].dropna().values.tolist()

FastMarkerCluster(restaurant_points).add_to(m)
FastMarkerCluster(delivery_points).add_to(m)

m

### Check for valid values

Many points are in the middle of the ocean.
We need to pick min and max coordinates. 

Use only the ones inside of India.

Remove the others

In [19]:
lat_min, lat_max = 6, 38
lon_min, lon_max = 68, 98

df = df[
    df["Restaurant_latitude"].between(lat_min, lat_max)
    & df["Restaurant_longitude"].between(lon_min, lon_max)
    & df["Delivery_location_latitude"].between(lat_min, lat_max)
    & df["Delivery_location_longitude"].between(lon_min, lon_max)
]

### Feature Extraction - distance

Use the Haversine distance, because latitude and longitude are geographic coordinates.

In [20]:
import numpy as np

def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in kilometers

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )

    c = 2 * np.arcsin(np.sqrt(a))

    return R * c

In [21]:
df["distance_km"] = haversine_distance(
    df["Restaurant_latitude"],
    df["Restaurant_longitude"],
    df["Delivery_location_latitude"],
    df["Delivery_location_longitude"]
)

In [22]:
df.head()

,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Type_of_order,Type_of_vehicle,Time_taken(min),distance_km
0,4607,INDORES13DEL02,37,4.9,22.745049,75.892471,22.765049,75.912471,Snack,motorcycle,24,3.025149
1,B379,BANGRES18DEL02,34,4.5,12.913041,77.683237,13.043041,77.813237,Snack,scooter,33,20.183530
2,5D6D,BANGRES19DEL01,23,4.4,12.914264,77.678400,12.924264,77.688400,Drinks,motorcycle,26,1.552758
3,7A6A,COIMBRES13DEL02,38,4.7,11.003669,76.976494,11.053669,77.026494,Buffet,motorcycle,21,7.790401
4,70A2,CHENRES12DEL01,32,4.6,12.972793,80.249982,13.012793,80.289982,Snack,scooter,30,6.210138


_**Disclaimer:** This material was partially based on previous classes and research given by Poliana N. Ferreira_